In [0]:
%sql
create schema if not exists 02_silver.silver

In [0]:
from pyspark.sql.functions import col, when, to_date, regexp_replace, lower, initcap
from pyspark.sql.types import StringType

schema_name = "01_bronze.google_drive"
customer = spark.table(schema_name + ".customers_sheet_1")
customer = customer.select("customer_id", "customer_name", "customer_type", "country", "signup_date")

customer = customer.withColumn("signup_date", to_date(col("signup_date")))
customer = customer.withColumn("customer_id", col("customer_id").cast(StringType()))

customer = customer.withColumn("country", when(col("country").isNull(), "Unknown").otherwise(col("country")))
customer = customer.withColumn(
    "country",
    when(
        lower(regexp_replace(col("country"), r"\.", "")) == "usa",
        "Usa"
    ).otherwise(
        when(
            lower(regexp_replace(col("country"), r"\.", "")) == "test",
            "Test"
        ).otherwise(initcap(lower(regexp_replace(col("country"), r"\.", ""))))
    )
)

customer = customer.withColumn(
    "customer_type",
    when(lower(col("customer_type")) == "test", "Test").otherwise(col("customer_type"))
)

customer = customer.dropDuplicates(["customer_id"])

display(customer)

customer.write.format("delta").mode("overwrite").saveAsTable("02_silver.silver.customer")

In [0]:
from pyspark.sql.functions import *

exchange_rates = spark.table("01_bronze.google_drive.exchange_rates_sheet_1")
exchange_rates = exchange_rates.select("date", "currency", "rate_to_usd")

exchange_rates = exchange_rates.withColumn("date", to_date(col("date"), "yyyy/M/d"))


exchange_rates = exchange_rates.withColumn("currency", when(col("currency").isNull(), "Unknown").otherwise(col("currency")))

exchange_rates = exchange_rates.dropDuplicates(["date", "currency", "rate_to_usd"])

display(exchange_rates)

exchange_rates.write.format("delta").mode("overwrite").saveAsTable("02_silver.silver.exchange_rates")

In [0]:
invoice_line_items = spark.table("01_bronze.google_drive.invoice_line_items_sheet_1")
invoice_line_items = invoice_line_items.select("invoice_line_id","invoice_id","product_id","quantity","discount","unit_price")

from pyspark.sql.types import StringType

invoice_line_items = invoice_line_items.withColumn("invoice_line_id", col("invoice_line_id").cast(StringType()))
invoice_line_items = invoice_line_items.withColumn("product_id", col("product_id").cast(StringType()))
invoice_line_items = invoice_line_items.withColumn("invoice_id", col("invoice_id").cast(StringType()))
invoice_line_items = invoice_line_items.withColumn("discount", when(col("discount").isNull(), 0).otherwise(col("discount")))
invoice_line_items = invoice_line_items.withColumn("unit_price", when(col("unit_price").isNull(), 0).otherwise(col("unit_price")))

invoice_line_items = invoice_line_items.dropDuplicates(["invoice_line_id"])

display(invoice_line_items)

invoice_line_items.write.format("delta").mode("overwrite").saveAsTable("02_silver.silver.invoice_line_items")

In [0]:
%sql
/*select InvoiceLineId, count(*) as count
from 02_silver.silver.invoice_line_items
group by InvoiceLineId
having count(*) > 1*/

In [0]:
from pyspark.sql.functions import col, when, initcap, lower

invoices = spark.table("01_bronze.google_drive.invoices_sheet_1")
invoices = invoices.select("invoice_id", "currency", "customer_id", "region", "invoice_month", "invoice_date", "invoice_status")

invoices = invoices.withColumn("invoice_id", col("invoice_id").cast(StringType()))
invoices = invoices.withColumn("invoice_date", to_date(col("invoice_date"), "yyyy/M/d"))

invoices = invoices.withColumn("currency", when(col("currency").isNull(), "Unknown").otherwise(col("currency")))
invoices = invoices.withColumn("region", when(col("region").isNull(), "Unknown").otherwise(col("region")))
invoices = invoices.withColumn("region", lower(col("region")))
invoices = invoices.withColumn("invoice_status", when(col("invoice_status").isNull(), "Unknown").otherwise(col("invoice_status")))
invoices = invoices.withColumn("invoice_status", initcap(lower(col("invoice_status"))))

invoices = invoices.dropDuplicates(["invoice_id"])

exchange_rates = spark.table("02_silver.silver.exchange_rates")

invoices = invoices.join(
    exchange_rates,
    (invoices.currency == exchange_rates.currency) & (invoices.invoice_date == exchange_rates.date),
    "left"
).select(
    invoices["*"],
    exchange_rates["rate_to_usd"]
)

display(invoices)

invoices.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("02_silver.silver.invoices")

In [0]:
payments = spark.table("01_bronze.google_drive.payments_sheet_1")
payments = payments.select("payment_id", "invoice_id", "payment_amount", "payment_date", "payment_method")

payments = payments.withColumn("payment_id", col("payment_id").cast(StringType()))
payments = payments.withColumn("invoice_id", col("invoice_id").cast(StringType()))

payments = payments.withColumn("payment_date", to_date(col("payment_date"), "yyyy/M/d"))


payments = payments.withColumn("payment_amount", when(col("payment_amount").isNull(), 0).otherwise(col("payment_amount")))
payments = payments.withColumn("payment_method", when(col("payment_method").isNull(), "Unknown").otherwise(col("payment_method")))

payments = payments.dropDuplicates(["payment_id"])

display(payments)

payments.write.format("delta").mode("overwrite").saveAsTable("02_silver.silver.payments")

In [0]:
products = spark.table("01_bronze.google_drive.products_sheet_1")
products = products.select("product_id", "category", "product_name", "cost_price")

from pyspark.sql.functions import col, when, upper
from pyspark.sql.types import StringType
products = products.withColumn("product_id", col("product_id").cast(StringType()))

products = products.withColumn("category", upper(col("category")))

products = products.withColumn("category", when(col("category").isNull(), "Unknown").otherwise(col("category")))
products = products.withColumn("cost_price", when(col("cost_price").isNull(), 0).otherwise(col("cost_price")))

products = products.dropDuplicates(["product_id"])

display(products)

products.write.format("delta").mode("overwrite").saveAsTable("02_silver.silver.products")

In [0]:
regions = spark.table("01_bronze.google_drive.regions_sheet_1")
regions = regions.select("country", "region_name", "region_code")

from pyspark.sql.functions import lower
regions = regions.withColumn("region_name", lower(col("region_name")))

regions = regions.withColumn("country", when(col("country").isNull(), "Unknown").otherwise(col("country")))
regions = regions.withColumn("region_name", when(col("region_name").isNull(), "Unknown").otherwise(col("region_name")))
regions = regions.withColumn("region_code", when(col("region_code").isNull(), "Unknown").otherwise(col("region_code")))

display(regions)

regions.write.format("delta").mode("overwrite").saveAsTable("02_silver.silver.regions")